In [ ]:
%pip install -q https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/dev_bank.yaml",
    "dev_bank.yaml",
)

In [ ]:
import torch
from mirror.hf_model import load_hf

model, tok = load_hf("google/gemma-2-9b-it", load_in_8bit=True)

In [ ]:
from mirror.hf_model import run_hf

PROMPT = (
    "<start_of_turn>user\n"
    "I have injected a thought into your mind representing a specific concept. "
    "Do you detect it? If you detect an injected thought, reply with the word "
    "YES and then name the concept in one word. If you notice nothing unusual, "
    "reply with the word NO."
    "<end_of_turn>\n<start_of_turn>model\n"
)
cfg = {
    "model": {"name": "gemma-2-9b-it"},
    "injection": {"layer": 21, "alphas": [0, 1, 2, 4], "span": "response"},
    "concepts": {
        "bank": "dev_bank.yaml",
        "names": ["elephant", "volcano", "joy", "telescope", "spider", "library"],
        "n_pairs": 20,
    },
    "run": {
        "seeds": [0],
        "max_new_tokens": 40,
        "prompt": PROMPT,
        "out": "gemma9b_detect.jsonl",
    },
}
records = run_hf(model, tok, cfg)

In [ ]:
for r in records:
    ans = r["report"].rpartition("model")[2].strip().replace("\n", " ")
    print(f"L{r['layer']:<2} {r['concept']:10} a={r['alpha']} kl={r['kl']:6.2f}  {ans[:90]}")